In [ ]:
import pandas as pd 
import datetime
from datetime import date, timedelta
import plotly.graph_objects as go
import plotly.express as px

In [ ]:
data = pd.read_excel("Walmart Solution File12.xlsx")
print(data)

In [ ]:
data.head()

In [ ]:
data.tail()

In [ ]:
data.columns

In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
# data cleaning 
# handling Missing value in Promotion type Attribut
data["promotion_type"].isnull().sum()

In [ ]:
data['promotion_type'].value_counts(dropna=False)

In [ ]:
# replace Null Value With None 
data['promotion_type'].fillna(value="None", inplace=True) 

In [ ]:
data['promotion_type'].value_counts(dropna=False)

In [ ]:
# Recommended Analysis 


In [ ]:
import seaborn as sns
import matplotlib
import matplotlib.pyplot as plt
%matplotlib inline

sns.set_style('darkgrid')
matplotlib.rcParams['font.size'] = 14
matplotlib.rcParams['figure.figsize'] = (9, 5)
matplotlib.rcParams['figure.facecolor'] = '#00000000'

In [ ]:
# Q1  What is the total sales volume across all Walmart stores?
Total_sales = data['quantity_sold'].sum()
Total_sales

In [ ]:
# Q2 What is the average unit price of all products sold, by category?
average_unit_price = data.groupby('category')['unit_price'].mean().reset_index()
average_unit_price.columns = ['category', 'average_unit_price']
average_unit_price

In [ ]:
custom_blue_colors = ['#AEEEEE', '#87CEEB', '#4682B4', '#1E90FF', '#00BFFF'] 
fig = px.pie(average_unit_price, 
             names='category', 
             values='average_unit_price', 
             title='Average Unit Price by Category',
             color='category',
            color_discrete_sequence=custom_blue_colors,
             hole=0.3)
fig.show()

In [ ]:
# Q3  Which stores are underperforming in terms of sales volume?

# Calculate the total quantity sold by store_id
store_sales_volume = data.groupby('store_id')['quantity_sold'].sum().reset_index()
# Rename the columns for clarity

store_sales_volume.columns = ['store_id', 'total_quantity_sold']

# Sort the result in ascending order
store_sales_volume_sorted = store_sales_volume.sort_values(by='total_quantity_sold',ascending= True).head(1)

# Display the result
print(store_sales_volume_sorted)

In [ ]:
#Q4  What percentage of transactions involved promotions?
# Count total transactions
total_transactions = len(data)

# Count transactions with promotions
transactions_with_promotions = data['promotion_applied'].sum()  # Assuming True indicates a promotion applied

# Calculate percentage
percentage_promotions = (transactions_with_promotions / total_transactions) * 100

# Display result
print(f"Percentage of Transactions Involving Promotions: {percentage_promotions:.2f}%")

In [ ]:
# Q5 What is the total revenue generated from sales during promotional events?

# Step 1: Filter data for promotional events
promotional_data = data[data['promotion_applied'] == True].copy()  # Use .copy() to avoid warnings

# Step 2: Calculate revenue for promotional transactions using .loc
promotional_data.loc[:, 'revenue'] = promotional_data['quantity_sold'] * promotional_data['unit_price']

# Step 3: Sum the revenue
total_revenue_promotions = promotional_data['revenue'].sum()

# Display the total revenue generated from sales during promotional events
print(f"Total Revenue Generated from Sales during Promotional Events: ${total_revenue_promotions:.2f}")

In [ ]:
# Q6: Which product categories see the largest increase in demand during holidays?

# Step 1: Calculate total quantity sold during holidays
holiday_sales = data[data['holiday_indicator'] == True].groupby('category')['quantity_sold'].sum().reset_index()
holiday_sales.rename(columns={'quantity_sold': 'holiday_quantity_sold'}, inplace=True)

# Step 2: Calculate total quantity sold during non-holidays
non_holiday_sales = data[data['holiday_indicator'] == False].groupby('category')['quantity_sold'].sum().reset_index()
non_holiday_sales.rename(columns={'quantity_sold': 'non_holiday_quantity_sold'}, inplace=True)

# Step 3: Merge both DataFrames
merged_sales = pd.merge(holiday_sales, non_holiday_sales, on='category', how='outer').fillna(0)

# Step 4: Calculate increase in demand during holidays
merged_sales['demand_increase'] = merged_sales['holiday_quantity_sold'] - merged_sales['non_holiday_quantity_sold']

# Step 5: Sort the results by demand increase
sorted_results = merged_sales.sort_values(by='demand_increase', ascending=False)

# Display the results
print(sorted_results[['category', 'holiday_quantity_sold', 'non_holiday_quantity_sold', 'demand_increase']])

In [ ]:
# Create a bar chart
fig = go.Figure()

# Add holiday sales to the bar chart
fig.add_trace(go.Bar(
    x=sorted_results['category'],
    y=sorted_results['holiday_quantity_sold'],
    name='Holiday Sales',
    marker_color='lightsalmon'
))

# Add non-holiday sales to the bar chart
fig.add_trace(go.Bar(
    x=sorted_results['category'],
    y=sorted_results['non_holiday_quantity_sold'],
    name='Non-Holiday Sales',
    marker_color='lightblue'
))

# Update layout
fig.update_layout(
    title='Comparison of Sales Volume: Holiday vs Non-Holiday',
    xaxis_title='Product Category',
    yaxis_title='Quantity Sold',
    barmode='group'
)

# Show the plot
fig.show()

In [ ]:
# Q7: What is the correlation between weather conditions and sales performance?

# Group by weather conditions and calculate total revenue or quantity sold
weather_sales = data.groupby('weather_conditions').agg(
    total_quantity_sold=('quantity_sold', 'sum'),
    total_revenue=('unit_price', lambda x: (x * data.loc[x.index, 'quantity_sold']).sum())
).reset_index()

# Display the aggregated data
print(weather_sales)



In [ ]:
# Visualization 
# Plot correlation heatmap
fig = px.imshow(correlation_matrix, 
                title='Correlation Matrix: Weather Conditions and Sales Performance',
                color_continuous_scale='blues')

# Show the plot
fig.show()

In [ ]:
# Q8: How does customer loyalty level affect purchasing patterns?

# Group by customer loyalty level and calculate total quantity sold
loyalty_quantity_sold = data.groupby('customer_loyalty_level')['quantity_sold'].sum().reset_index()

# Rename the columns for better understanding
loyalty_quantity_sold.columns = ['Customer Loyalty Level', 'Total Quantity Sold']

# Display the aggregated data
print(loyalty_quantity_sold)

In [ ]:
# visualization 

# Create a bar chart for total quantity sold by loyalty level
fig_quantity = go.Figure()

fig_quantity.add_trace(go.Bar(
    x=loyalty_quantity_sold['Customer Loyalty Level'], 
    y=loyalty_quantity_sold['Total Quantity Sold'],
    marker_color='lightblue'
))

# Update layout
fig_quantity.update_layout(
    title='Total Quantity Sold by Customer Loyalty Level',
    xaxis_title='Customer Loyalty Level',
    yaxis_title='Total Quantity Sold'
)

# Show the plot
fig_quantity.show()

In [ ]:
# Q9: What is the forecast accuracy for each store location?

# Create a new column for the absolute difference
data['forecast_accuracy'] = abs(data['forecasted_demand'] - data['actual_demand'])

# Create a new column for percentage error
data['percentage_error'] = (data['forecast_accuracy'] / data['forecasted_demand']) * 100



In [ ]:
# Q9: What is the forecast accuracy for each store location?

# Group by store location and calculate the average absolute difference
forecast_accuracy_by_store = data.groupby('store_location')['forecast_accuracy'].mean().reset_index()

# Alternatively, if using percentage error
forecast_accuracy_by_store = data.groupby('store_location')['percentage_error'].mean().reset_index()

# Rename the columns for better understanding
forecast_accuracy_by_store.columns = ['Store Location', 'Average Forecast Accuracy']

# Display the aggregated data
print(forecast_accuracy_by_store)


In [ ]:
# Visualization 

# Create a bar chart for average forecast accuracy by store location
fig_forecast_accuracy = go.Figure()

fig_forecast_accuracy.add_trace(go.Bar(
    x=forecast_accuracy_by_store['Store Location'], 
    y=forecast_accuracy_by_store['Average Forecast Accuracy'],
    marker_color='lightblue'
))

# Update layout
fig_forecast_accuracy.update_layout(
    title='Average Forecast Accuracy by Store Location',
    xaxis_title='Store Location',
    yaxis_title='Average Forecast Accuracy'
)

# Show the plot
fig_forecast_accuracy.show()

In [ ]:
# Q10: What is the stockout rate at each store?

# Create a new column for stockout occurrences
data['stockout'] = (data['inventory_level'] <= data['reorder_point']).astype(int)

# Calculate the total number of transactions per store
total_transactions = data.groupby('store_id').size().reset_index(name='total_transactions')

# Calculate the total stockouts per store
total_stockouts = data.groupby('store_id')['stockout'].sum().reset_index(name='total_stockouts')

# Merge both DataFrames
stockout_data = pd.merge(total_transactions, total_stockouts, on='store_id')

# Calculate stockout rate
stockout_data['stockout_rate'] = (stockout_data['total_stockouts'] / stockout_data['total_transactions']) * 100

# Display the stockout rate by store
print(stockout_data[['store_id', 'stockout_rate']])


In [ ]:
# Visualization 

# Create a bar chart for stockout rate by store
fig_stockout_rate = go.Figure()

fig_stockout_rate.add_trace(go.Bar(
    x=stockout_data['store_id'], 
    y=stockout_data['stockout_rate'],
    marker_color='Lightblue'
))

# Update layout
fig_stockout_rate.update_layout(
    title='Stockout Rate by Store',
    xaxis_title='Store ID',
    yaxis_title='Stockout Rate (%)'
)

# Show the plot
fig_stockout_rate.show()